# Memory coalescing and bandwidth-bound kernels: copy, stride and transpose

How a warp's 32 loads become 32-byte memory sectors, why strided access can waste most of your bandwidth, how vectorised loads help, and how to write a matrix transpose that reads and writes efficiently using shared memory without bank conflicts.

This notebook holds the code from the lesson **[Memory coalescing and bandwidth-bound kernels: copy, stride and transpose](https://inferenceacademy.com/learn/kern-cuda/memory-coalescing-and-bandwidth-bound-kernels)** on Inference Academy, in the same order. Read the lesson for the explanations, diagrams and exercises, and run the cells here as you go.

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*, then run the first cell to check the GPU.

In [ ]:
!nvidia-smi

## How a warp reads memory

## Lab: measure it

In [ ]:
%%writefile stride.cu
// stride.cu
#include <cstdio>
__global__ void copy_strided(const float* in, float* out, int n, int stride) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  long j = (long)i * stride % n;           // wrap so every stride touches n elements
  if (i < n) out[i] = in[j];
}
int main() {
  const int n = 1 << 25;                   // 32M floats, 128 MB
  float *in, *out; cudaMalloc(&in, n * 4); cudaMalloc(&out, n * 4);
  cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
  for (int stride : {1, 2, 4, 8, 16, 32}) {
    copy_strided<<<n / 256, 256>>>(in, out, n, stride);
    cudaEventRecord(s);
    for (int r = 0; r < 10; r++) copy_strided<<<n / 256, 256>>>(in, out, n, stride);
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms; cudaEventElapsedTime(&ms, s, e); ms /= 10;
    printf("stride %2d: %.3f ms, useful %.0f GB/s\n", stride, ms, 2.0 * n * 4 / ms / 1e6);
  }
}

In [ ]:
!nvcc -O3 -arch=sm_75 -o stride stride.cu && ./stride

## Vectorised loads

Snippet from the lesson (add it to your program; not run on its own):

```cpp
__global__ void copy_vec4(const float4* in, float4* out, int n4) {
  for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n4; i += blockDim.x * gridDim.x)
    out[i] = in[i];                         // one 16-byte load, one 16-byte store
}
// n4 = n / 4. The pointers must be 16-byte aligned (cudaMalloc pointers are).
```

## Transpose: the classic problem

Snippet from the lesson (add it to your program; not run on its own):

```cpp
// in is rows x cols, out is cols x rows, both row-major
__global__ void transpose_naive(const float* in, float* out, int rows, int cols) {
  int c = blockIdx.x * blockDim.x + threadIdx.x;   // column, fastest-varying across a warp
  int r = blockIdx.y * blockDim.y + threadIdx.y;
  if (r < rows && c < cols) out[c * rows + r] = in[r * cols + c];
}
```

### The fix: stage through shared memory

Snippet from the lesson (add it to your program; not run on its own):

```cpp
#define TILE 32
#define ROWS 8   // each thread handles TILE / ROWS = 4 elements

__global__ void transpose_smem(const float* in, float* out, int rows, int cols) {
  __shared__ float tile[TILE][TILE + 1];   // +1 avoids bank conflicts, explained below
  int x = blockIdx.x * TILE + threadIdx.x;
  int y = blockIdx.y * TILE + threadIdx.y;
  for (int k = 0; k < TILE; k += ROWS)
    if (x < cols && y + k < rows) tile[threadIdx.y + k][threadIdx.x] = in[(y + k) * cols + x];
  __syncthreads();
  x = blockIdx.y * TILE + threadIdx.x;     // swap block coordinates for the output
  y = blockIdx.x * TILE + threadIdx.y;
  for (int k = 0; k < TILE; k += ROWS)
    if (x < rows && y + k < cols) out[(y + k) * rows + x] = tile[threadIdx.x][threadIdx.y + k];
}
// launch: dim3 block(TILE, ROWS); dim3 grid((cols + TILE - 1) / TILE, (rows + TILE - 1) / TILE);
```

### Bank conflicts

## How fast should each version be?

## Where this shows up in inference

## What you can now do

## Common mistakes

## Exercise

## Going deeper

---
Code from [Inference Academy](https://inferenceacademy.com), MIT licensed. Lesson: https://inferenceacademy.com/learn/kern-cuda/memory-coalescing-and-bandwidth-bound-kernels